In [1]:
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from mtcnn import MTCNN  # MTCNN from the mtcnn library

# Initialize MTCNN detector
detector = MTCNN()

# Define transformation (resize and normalize for ImageNet models)
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),  # Match your model's input size
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),  # ImageNet normalization
    ]
)

# Load multiple saved models
model_paths = [
    r"D:\Mtech\sem1\ML\deep fake\code\experiments\deepfake_vgg16_20241120_203441\checkpoints\best_model.pth",
    r"D:\Mtech\sem1\ML\deep fake\code\experiments\deepfake_xceptionnet_20241121_124840\checkpoints\best_model.pth",
]
models = []

for path in model_paths:
    model = torch.load(path)  # Load the model
    model.eval()  # Set to evaluation mode
    models.append(model)


def detect_and_crop_face(image_path):
    """Detect and crop the face using MTCNN."""
    # Load the image
    image = Image.open(image_path).convert("RGB")

    # Detect faces using MTCNN
    detection_results = detector.detect(image)

    # If no face is detected, handle the error
    if detection_results[0] is None:
        raise ValueError("No face detected in the image.")

    # Get the bounding box of the largest face detected
    boxes = detection_results[0]
    largest_box = max(
        boxes, key=lambda box: (box[2] - box[0]) * (box[3] - box[1])
    )  # Largest area
    x1, y1, x2, y2 = map(int, largest_box)

    # Crop the face
    cropped_face = image.crop((x1, y1, x2, y2))

    return cropped_face


def classify_image(image_path, models):
    """Classify an image using multiple models."""
    try:
        # Detect and crop the face
        cropped_face = detect_and_crop_face(image_path)
    except ValueError as e:
        print(e)
        return None, []

    # Apply transformations
    transformed_image = transform(cropped_face).unsqueeze(0)  # Add batch dimension

    # Get predictions from all models
    predictions = []
    with torch.no_grad():
        for model in models:
            outputs = model(transformed_image)
            logits = (
                outputs if not isinstance(outputs, torch.nn.Module) else outputs.logits
            )
            predicted_class = torch.argmax(
                logits, dim=1
            ).item()  # 0 for Real, 1 for Fake
            predictions.append(predicted_class)

    return cropped_face, predictions


# Classify an image and display results
image_path = "../data\test_data\fake\5.jpg"  # Replace with your input image path
cropped_face, predictions = classify_image(image_path, models)

if cropped_face:
    # Display the cropped face and predictions
    plt.imshow(cropped_face)
    plt.axis("off")
    plt.title(f"Predictions: {predictions}")  # Adjust based on your classes: Real/Fake
    plt.show()
else:
    print("No face detected, unable to classify.")


C:\Users\yashs\AppData\Local\Temp\ipykernel_14584\3959660981.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(path)  # Load the model


AttributeError: 'dict' object has no attribute 'eval'

In [2]:
pip install facenet-pytorch

^C
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms, models
from mtcnn import MTCNN
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
from torchvision.models import vgg16_bn, VGG16_BN_Weights
import seaborn as sns
import logging

# Configure logging to suppress TensorFlow warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)

class DeepFakeDetector(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(DeepFakeDetector, self).__init__()
        
        # Load the vgg16 model with Batch Normalization
        self.vgg16 = vgg16_bn(weights=VGG16_BN_Weights.IMAGENET1K_V1)

        if freeze_backbone:
            for param in self.vgg16.features.parameters():
                param.requires_grad = False

        dropout_rate = 0.5
        num_classes = 2  # Binary classification
        
        # Modifying classifier with custom architecture
        self.vgg16.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),  # First layer
            nn.ReLU(inplace=True),
            nn.LayerNorm(4096),  # Layer Normalization for regularization
            nn.Dropout(dropout_rate),  # Dropout for regularization
            nn.Linear(4096, 2048),  # Second layer with reduced dimensions
            nn.ReLU(inplace=True),
            nn.LayerNorm(2048),  # Additional Layer Normalization
            nn.Dropout(dropout_rate),  # Dropout again
            nn.Linear(2048, 1024),  # Third layer
            nn.ReLU(inplace=True),
            nn.LayerNorm(1024),  # Additional Layer Normalization
            nn.Dropout(dropout_rate),  # Dropout again
            nn.Linear(1024, num_classes),  # Output layer
        )

        # Initialize weights of added layers
        self._initialize_weights()

    def forward(self, x):
        return self.vgg16(x)

    def _initialize_weights(self):
        """Custom weight initialization for added layers."""
        for m in self.vgg16.classifier:
            if isinstance(m, nn.Linear):
                init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    init.constant_(m.bias, 0)

class DeepFakeDetectionPipeline:
    def __init__(self, model_paths, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        
        # Initialize face detector with MTCNN
        self.face_detector = MTCNN()
        
        # Define padding parameters
        self.padding = 40
        self.extra_top = 40
        
        # Load multiple models
        self.models = self._load_models(model_paths)
        
        # Define image transformations
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def _load_models(self, model_paths):
        """Load models from provided paths"""
        models = {}
        for name, path in model_paths.items():
            try:
                # Load the full checkpoint
                checkpoint = torch.load(path, map_location=self.device)
                
                # Initialize the model architecture
                model = DeepFakeDetector(freeze_backbone=True)
                
                # Load the model state dict from the checkpoint
                model.load_state_dict(checkpoint['model_state_dict'])
                model.to(self.device)
                model.eval()
                
                models[name] = model
            except Exception as e:
                print(f"Error loading model {name}: {str(e)}")
                continue
        return models

    def extract_faces(self, image_path, min_confidence=0.95):
        """Extract faces from image using MTCNN with padding"""
        try:
            # Read image
            if isinstance(image_path, str):
                frame = cv2.imread(image_path)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            else:
                frame = image_path if len(image_path.shape) == 3 else cv2.cvtColor(image_path, cv2.COLOR_BGR2RGB)
            
            detections = self.face_detector.detect_faces(frame)
            faces = []
            locations = []
            
            height, width = frame.shape[:2]
            
            if detections:
                best_detection = max(detections, key=lambda x: x["confidence"])
                
                if best_detection["confidence"] > min_confidence:
                    x, y, w, h = best_detection['box']
                    
                    x_new = max(0, x - self.padding)
                    y_new = max(0, y - self.padding - self.extra_top)
                    w_new = min(width - x_new, w + 2 * self.padding)
                    h_new = min(height - y_new, h + 2 * self.padding + self.extra_top)
                    
                    face_img = frame[y_new:y_new+h_new, x_new:x_new+w_new]
                    face_pil = Image.fromarray(face_img)
                    
                    faces.append(face_pil)
                    locations.append([x_new, y_new, x_new+w_new, y_new+h_new])
            
            return faces, locations
            
        except Exception as e:
            print(f"Error in face extraction: {str(e)}")
            return [], []
    
    def preprocess_face(self, face):
        """Apply transformations to extracted face"""
        try:
            return self.transform(face).unsqueeze(0).to(self.device)
        except Exception as e:
            print(f"Error in preprocessing: {str(e)}")
            return None
    
    def predict_single_model(self, model, face_tensor):
        """Get prediction from a single model"""
        with torch.no_grad():
            outputs = model(face_tensor)
            probs = F.softmax(outputs, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred_class].item()
            return pred_class, confidence

    def analyze_image(self, image_path, visualization_path=None):
        """Complete analysis pipeline for an image"""
        faces, locations = self.extract_faces(image_path)
        
        if not faces:
            print("No faces detected in the image!")
            return None
        
        results = []
        
        for idx, face in enumerate(faces):
            face_tensor = self.preprocess_face(face)
            if face_tensor is None:
                continue
            
            face_results = {
                'location': locations[idx],
                'predictions': {}
            }
            
            for model_name, model in self.models.items():
                pred_class, confidence = self.predict_single_model(model, face_tensor)
                face_results['predictions'][model_name] = {
                    'class': 'FAKE' if pred_class == 1 else 'REAL',
                    'confidence': confidence
                }
            
            results.append(face_results)
        
        if visualization_path and results:
            self.visualize_results(image_path, results, visualization_path)
        
        return results
    
    def visualize_results(self, image_path, results, output_path):
        """Create visualization of detection results"""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        fig = plt.figure(figsize=(20, 10 * len(results)))
        
        for idx, result in enumerate(results):
            ax1 = plt.subplot(len(results), 2, 2*idx + 1)
            ax1.imshow(img)
            
            box = result['location']
            rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                               fill=False, color='red', linewidth=2)
            ax1.add_patch(rect)
            ax1.axis('off')
            ax1.set_title('Detected Face')
            
            ax2 = plt.subplot(len(results), 2, 2*idx + 2)
            
            models = list(result['predictions'].keys())
            confidences = [pred['confidence'] * 100 for pred in result['predictions'].values()]
            colors = ['red' if pred['class'] == 'FAKE' else 'green' 
                     for pred in result['predictions'].values()]
            
            bars = ax2.barh(models, confidences, color=colors)
            
            for bar in bars:
                width = bar.get_width()
                ax2.text(width, bar.get_y() + bar.get_height()/2,
                        f'{width:.1f}%', ha='left', va='center', fontsize=10)
            
            ax2.set_xlim(0, 100)
            ax2.set_title(f'Model Predictions (Face {idx+1})')
            ax2.set_xlabel('Confidence (%)')
        
        plt.tight_layout()
        plt.savefig(output_path)
        plt.close()

def main():
    # Model paths
    model_paths = {
        'VGG16': r'D:\Mtech\sem1\ML\deep fake\code\experiments\deepfake_vgg16_20241120_203441\checkpoints\best_model.pth'
    }
    
    # Initialize pipeline
    pipeline = DeepFakeDetectionPipeline(model_paths)
    
    # Process single image
    # image_path = r"D:\Mtech\sem1\ML\deep fake\data\test_data\fake\6.jpg"
    image_path = r"D:\Mtech\sem1\ML\deep fake\data\test_data\1.jpg" # real image
    output_path = 'detection_results.jpg'
    
    results = pipeline.analyze_image(image_path, output_path)
    
    if results:
        print("\nDetection Results:")
        for idx, result in enumerate(results):
            print(f"\nFace {idx+1}:")
            for model_name, pred in result['predictions'].items():
                print(f"{model_name}: {pred['class']} "
                      f"(Confidence: {pred['confidence']*100:.2f}%)")

if __name__ == "__main__":
    main()

C:\Users\yashs\AppData\Local\Temp\ipykernel_20052\3344734933.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=self.device)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step

Detection Results:

Face 1:
VGG16: FAKE (Confidence: 100.00%)
